In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys
import seaborn as sns
from tqdm import tqdm
import networkx as nx

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/OtherLibraries/pyedm4hep")
from pyedm4hep import EDM4hepEvent, EDM4hepEventBatch

## Roadmap

1. Load an edm4hep file
2. Load the particles and tracker hits


## Loading

In [3]:
# Load the edm4hep file with uproot
edm_input_file = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/full_pileup/ttbar/v1/runs/0/edm4hep.root"
root_file = uproot.open(edm_input_file)["events"]

# Print the keys of the root file
print(root_file.keys())


['ECalBarrelCollection', 'ECalBarrelCollection/ECalBarrelCollection.cellID', 'ECalBarrelCollection/ECalBarrelCollection.energy', 'ECalBarrelCollection/ECalBarrelCollection.position.x', 'ECalBarrelCollection/ECalBarrelCollection.position.y', 'ECalBarrelCollection/ECalBarrelCollection.position.z', 'ECalBarrelCollection/ECalBarrelCollection.contributions_begin', 'ECalBarrelCollection/ECalBarrelCollection.contributions_end', '_ECalBarrelCollection_contributions', '_ECalBarrelCollection_contributions/_ECalBarrelCollection_contributions.index', '_ECalBarrelCollection_contributions/_ECalBarrelCollection_contributions.collectionID', 'ECalBarrelCollectionContributions', 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.PDG', 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.energy', 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.time', 'ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.stepPosition.x', 'ECalBarrelColl

In [ ]:
root_file[]

AttributeError: 'Model_TTree_v20' object has no attribute 'ECalBarrelCollectionContributions'

In [ ]:
ECalBarrelCollectionContributions/ECalBarrelCollectionContributions.stepPosition.x

In [3]:
edm_input_file = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/full_pileup/ttbar/v1/runs/0/edm4hep.root"
batch = EDM4hepEventBatch(edm_input_file, events=(0, 16), full_load=False)

In [4]:
hits_batch = batch.get_tracker_hits_df()
# Print the amount of memory used by the hits_batch
print(f"Memory used by the hits_batch: {hits_batch.memory_usage().sum() / 1024 / 1024:.2f} MB")

Memory used by the hits_batch: 6.35 MB


In [5]:
hits_batch

,event_id,subentry,cellID,time,x,y,z,detector,particle_id,r,R,phi,theta,eta
0,0,0,242237027913222,1.748552,24.238054,-20.907667,-499.783925,PixelBarrelReadout,532,32.009590,500.807933,-0.711762,3.077633,-3.442311
1,0,1,2747873167638,1.344684,-10.544569,67.779596,-396.823997,PixelBarrelReadout,637,68.594909,402.709009,1.725131,2.970424,-2.455809
2,0,2,8935645972758,1.329044,-19.084869,64.983989,-392.570268,PixelBarrelReadout,636,67.728510,398.369887,1.856450,2.970749,-2.457715
3,0,3,276868490136086,1.362648,-19.975461,66.440722,-402.136339,PixelBarrelReadout,636,69.378589,408.077228,1.862850,2.970749,-2.457720
4,0,4,242234780093446,0.474317,-4.297228,-32.705119,-137.362575,PixelBarrelReadout,235,32.986224,141.267718,-1.701441,2.905915,-2.133796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64579,15,805,880469442878,8.835363,-942.500247,-31.734169,2270.500000,LongStripEndcapReadout,7861,943.034343,2458.553238,-3.107935,0.393662,1.612378
64580,15,806,996432511294,8.858586,-945.969513,-33.881429,2275.500000,LongStripEndcapReadout,7861,946.576078,2464.529675,-3.105791,0.394211,1.610947
64581,15,807,867583443230,30.983475,-29.814803,963.090931,1625.427851,LongStripEndcapReadout,7877,963.552316,1889.563114,1.601744,0.535108,1.294165
64582,15,808,627066445838,19.469742,581.036841,-549.408579,1290.489829,LongStripEndcapReadout,7882,799.658425,1518.162572,-0.757427,0.554746,1.256275


In [5]:
particles = batch.get_particles_df()

Augmenting particle hit counts with tracker hits


In [6]:
particles

,event_id,subentry,PDG,simulatorStatus,charge,time,mass,vx,vy,vz,...,pz,parents_begin,parents_end,vr,energy,kinetic_energy,particle_id,created_in_simulation,num_tracker_hits,num_calo_hits
0,0,0,2212,0,1.000000,0.000000,0.938270,0.000000,0.000000,0.000000,...,6999.999937,0,0,0.000000,7000.000000,6999.061730,0,False,0,0
1,0,1,21,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,385.000495,0,1,0.000000,385.000495,385.000495,1,False,0,0
2,0,2,2,0,0.666667,0.000000,0.330000,0.000000,0.000000,0.000000,...,1244.573976,1,2,0.000000,1244.574020,1244.244020,2,False,0,0
3,0,3,2103,0,0.333333,0.000000,0.771330,0.000000,0.000000,0.000000,...,5370.425347,2,3,0.000000,5370.425402,5369.654072,3,False,0,0
4,0,4,2212,0,1.000000,0.000000,0.938270,0.000000,0.000000,0.000000,...,-6999.999937,3,3,0.000000,7000.000000,6999.061730,4,False,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111116,15,7993,22,1493172224,0.000000,10.801742,0.000000,209.678947,257.640052,3221.196893,...,0.114526,9084,9085,332.180158,0.115407,0.115407,7993,True,0,0
111117,15,7994,22,1493172224,0.000000,10.817452,0.000000,210.172701,258.026332,3225.864750,...,0.146315,9085,9086,332.791455,0.147569,0.147569,7994,True,0,0
111118,15,7995,-11,1224736768,1.000000,11.038603,0.000511,216.498735,263.890215,3291.600581,...,0.106835,9086,9087,341.335242,0.107750,0.107239,7995,True,0,0
111119,15,7996,-11,1224736768,1.000000,10.833244,0.000511,210.603623,258.348389,3230.569053,...,0.102825,9087,9088,333.313330,0.103660,0.103149,7996,True,0,0


In [6]:
hits_batch.event_id.unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])